# 📝 평가셋 구축 과제 — 내 손으로 만들고, 그 평가셋으로 재기

> 지금까지는 **누군가 만들어 둔 평가셋**으로 검색을 쟀습니다. 이번에는 그 평가셋을 **여러분이 만듭니다.** 문서를 읽고 질문을 쓰고 정답 라벨을 붙인 뒤, 그것이 쓸 만한지 점검하고, 그 평가셋으로 실제 검색기를 재어 **어디가 문제인지 찾아 고치는** 데까지 갑니다.

## 이 과제의 자료
`data/hr_faq.csv` — 사내 인사·복리후생 FAQ **24건**입니다. 한 행이 **한 문서**이고, 한 건이 세 문장이라 **자를 것이 없습니다.** 그래서 정답 라벨도 조각 id 가 아니라 FAQ 의 id(`h01`·`h02` …) 그대로입니다. 실습을 위해 만든 **가상의 사내 규정**이니 내용을 실제 제도로 받아들이지는 마세요.

## 풀이 방법
1. 위에서부터 순서대로 실행하세요. **뒤 문제가 앞 문제의 결과를 그대로 씁니다.**
2. 각 **답안 셀**에 코드를 채우고 **자가채점 셀**로 확인하세요.

- **모델을 한 번도 부르지 않습니다** — `OPENAI_API_KEY` 가 필요 없습니다. 임베딩과 검색만 쓰므로 값이 언제나 같습니다.
- **4번부터는 정답이 사람마다 다릅니다.** 여러분이 쓴 질문과 라벨이 곧 답이니까요. 그래서 자가채점은 값이 아니라 **평가셋이 갖춰야 할 성질**을 봅니다 — 정답 id 가 실제로 있는가, 근거 문장이 원문에 있는가, 지표가 앞뒤가 맞는가.

화이팅!

## 1. 재는 대상 살펴보기

**배경**: 평가셋을 만들기 전에 **무엇을 재는지**부터 알아야 합니다. 어떤 주제가 몇 건씩 들어 있는지, 한 건이 얼마나 긴지 보지 않고 질문부터 쓰면, 문서에 없는 것을 묻거나 한 주제만 잔뜩 묻게 됩니다.

**요구사항**:
- `data/hr_faq.csv` 를 읽어 변수 **`faq`** 에 담으세요.
- 문서 수를 `문서 24건` 형태로 출력하세요.
- `display()` 로 **앞 3행**을 보세요.
- `분류` 열의 값별 개수를 출력하세요.
- 본문 평균 글자 수를 **정수로** 출력하세요.

**예시**

```
문서 24건
(앞 3행 표)
분류
휴가      8
...
본문 평균 글자 수: 101
```

<details><summary>힌트</summary>

```text
접근방법:
- 읽고, 세고, 눈으로 본다. 새로운 것은 없다.

세부구현:
1. pandas 로 CSV 를 읽는다.
2. 행 수는 len 으로 센다.
3. 분류별 개수는 값 세기 메서드로, 표로 찍히니 문자열로 바꿔 출력한다.
4. 본문 길이는 문자열 길이 accessor 로 재고 평균을 낸 뒤 정수로 바꾼다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(faq) == 24, 'hr_faq.csv 를 읽어 faq 에 담으세요'
assert list(faq.columns) == ['id', '분류', '제목', '본문'], '열 이름이 다릅니다'
assert faq['id'].is_unique, '문서 id 는 유일해야 합니다'
print('✅ 통과!')

## 2. 색인 만들기 — 한 행이 한 문서

**배경**: 검색을 재려면 검색기가 있어야 합니다. 자를 것이 없으니 **한 행을 그대로 문서 하나로** 감싸 색인합니다. 평가에서 필요한 것은 검색 결과의 본문이 아니라 **id** 이므로(정답 라벨이 id 니까요) id 만 꺼내 주는 함수를 만들어 둡니다.

**요구사항**:
- 각 행을 `Document` 로 감싸 리스트 **`documents`** 를 만드세요.
  - `page_content` 는 **제목과 본문을 줄바꿈(`\n`)으로 이은 문자열**입니다(제목도 검색에 도움이 됩니다).
  - `metadata` 는 **`{'doc_id': 그 행의 id}`** 입니다.
- 임베딩은 **`jhgan/ko-sroberta-multitask`** 를 쓰고, `Chroma.from_documents` 로 색인 **`store`** 를 만드세요. `collection_name` 은 **`'hr_faq'`**, `ids=` 에는 FAQ 의 **id 목록을 그대로** 넘깁니다.
- 함수 **`search_ids(query, k) -> list`** 를 정의하세요. 질문과 가장 가까운 문서 `k`개의 **`doc_id` 를 1위부터 순서대로** 담은 리스트를 돌려줍니다.

**예시**

```
search_ids('연말정산은 언제 하나요?', 3)   -> ['h17', 'h16', 'h23']   (사람마다 뒤 순위는 다를 수 있습니다)
```

> 색인을 만들 때 임베딩 모델을 내려받느라 **처음 한 번은 잠시 걸립니다.** 이 셀은 한 번만 실행하고 끝까지 재사용하세요.

<details><summary>힌트</summary>

```text
접근방법:
- 자를 것이 없으니 반복문 한 번으로 문서 리스트가 끝난다.
- 검색기는 색인을 감싸 만들고, 결과에서 꼬리표만 꺼낸다.

세부구현:
1. 행을 돌며 제목과 본문을 줄바꿈으로 이어 page_content 로, id 를 꼬리표로 넣어 Document 를 만든다.
2. 임베딩 객체를 만들고 색인에 문서 리스트를 넣는다. 이때 id 목록도 함께 넘긴다.
3. 함수 안에서 색인을 검색기로 바꾸되 상위 몇 개를 볼지는 인자로 받은 값을 쓴다.
4. 검색 결과 하나하나에서 꼬리표의 doc_id 만 꺼내 리스트로 돌려준다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
found = search_ids('재택근무는 며칠까지 신청할 수 있나요?', 3)
assert isinstance(found, list) and len(found) == 3, '문서 id 3개를 담은 리스트를 돌려주세요'
assert all(f in set(faq['id']) for f in found), 'doc_id 만 담아야 합니다(본문이 아니라)'
assert len(set(found)) == 3, '같은 문서가 두 번 들어왔습니다 - ids= 를 넘겼는지 확인하세요'
assert 'h10' in found, '재택근무 문서가 상위 3개에 없습니다 - page_content 를 확인하세요'
print('✅ 통과!')

## 3. 근거 문장을 검증하는 도구 만들기

**배경**: 라벨에 **근거 문장**을 함께 적어 두면 나중에 사람이 검수할 수 있고, 색인을 다시 만들어도 그 문장을 찾아 라벨을 다시 붙일 수 있습니다. 그런데 사람이 옮겨 적다 보면 원문과 조금씩 달라집니다. **줄바꿈·띄어쓰기 차이는 넘어가되, 원문에 없는 문장은 걸러야** 합니다. 4번에서 여러분이 적을 근거를 5번에서 이 도구로 검사할 것입니다.

이 도구가 보는 것은 **그 문장이 원문에 있나** 하나뿐입니다. 그 문장이 **질문의 답인지는 보지 못합니다** — 같은 문서의 엉뚱한 문장을 그대로 옮겨 와도 통과합니다. 그래서 검증은 명백한 오류만 걷어 내고, **라벨이 옳은지는 사람이 읽고 정합니다.**

**요구사항**: 두 함수와 사전 하나를 만드세요.
- **`body`** — 문서 id 로 본문을 바로 꺼낼 수 있는 사전(`{'h01': '입사 첫해에는 ...', ...}`).
- **`squeeze(text) -> str`** — 공백(띄어쓰기·줄바꿈)을 **모두** 지운 문자열을 돌려줍니다.
- **`quoted_in_faq(sentence, doc_id) -> bool`** — 그 문장이 `doc_id` 문서의 **본문 안에 실제로 있으면** `True`, 아니면 `False`. 공백 차이는 무시합니다.
  - **빈 문자열이나 공백뿐인 문장은 `False`** 입니다. 공백을 지우면 빈 문자열이 되는데, 빈 문자열은 어떤 본문에도 '들어 있다'가 되어 버리기 때문입니다.

**예시**

```
squeeze(' 연차는  하루\n단위 ')                              -> '연차는하루단위'
quoted_in_faq('남은 연차는 다음 해로 이월되지 않습니다', 'h01')   -> True
quoted_in_faq('남은 연차는 수당으로 지급됩니다', 'h01')          -> False   (원문에 없는 문장)
quoted_in_faq('', 'h01')                                  -> False
quoted_in_faq('연차는 회계연도를 기준으로 매년 1월 1일에 새로 부여됩니다', 'h01')
                                                          -> True    (원문에 있기만 하면 True 다.
                                                                      '이월되나요?' 의 답이 아니어도 통과한다)
```

<details><summary>힌트</summary>

```text
접근방법:
- 공백만 지우고 나면 '들어 있나'는 부분문자열 검사 한 줄이다.
- 빈 문자열을 먼저 걸러야 한다. 안 그러면 무엇이든 통과한다.

세부구현:
1. id 와 본문을 짝지어 사전으로 만든다.
2. 정규식으로 공백 문자를 전부 빈 문자열로 바꾸는 함수를 만든다.
3. 검증 함수는 먼저 문장의 앞뒤 공백을 떼고 비어 있으면 바로 거짓을 돌려준다.
4. 비어 있지 않으면 양쪽 모두 공백을 지운 뒤 포함 여부를 돌려준다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert squeeze(' 연차는  하루\n단위 ') == '연차는하루단위'
assert quoted_in_faq('남은 연차는 다음 해로 이월되지 않습니다', 'h01') is True
# 줄바꿈·띄어쓰기만 다른 인용은 통과시켜야 한다
assert quoted_in_faq('남은 연차는  다음 해로\n이월되지 않습니다', 'h01') is True
assert quoted_in_faq('남은 연차는 수당으로 지급됩니다', 'h01') is False, '원문에 없는 문장입니다'
assert quoted_in_faq('', 'h01') is False, '빈 문자열은 False 여야 합니다'
assert quoted_in_faq('   ', 'h01') is False, '공백뿐인 문장도 False 여야 합니다'
assert len(body) == 24 and body['h01'].startswith('입사 첫해'), 'body 는 id 로 본문을 꺼내는 사전입니다'
print('✅ 통과!')

## 4. 평가셋 만들기 — 질문 8개에 라벨 붙이기

**배경**: 이제 **재는 도구**를 만듭니다. 평가셋 한 문항은 세 가지입니다 — **질문**, 그 답이 들어 있는 **문서 id**, 그리고 그 문서에서 답이 되는 **문장**. 앞의 둘만 있어도 점수는 나오지만, 근거 문장이 없으면 나중에 그 라벨이 맞는지 아무도 확인할 수 없습니다.

**요구사항**: FAQ 24건을 직접 읽고 질문 **8개 이상**을 써서 DataFrame **`my_eval`** 에 담으세요.

- 열 이름과 순서는 **`query_id`, `query`, `gold_ids`, `evidence`** 입니다.
  - `query_id` — `q1` 부터 순서대로
  - `query` — 사내 직원이 실제로 물어볼 법한 말투의 질문
  - `gold_ids` — 정답 문서 id 를 **`'|'`** 로 이어 붙인 문자열 (예: `'h04|h05'`)
  - `evidence` — 근거 문장을 **`' || '`**(공백-파이프-파이프-공백)로 이어 붙인 문자열. **`gold_ids` 와 같은 순서, 같은 개수**여야 합니다.
- 근거 문장은 **본문에서 그대로** 떼어 오세요(요약하거나 고쳐 쓰면 5번 검증에서 걸립니다).
- **원문 문장을 그대로 복사해 질문으로 만들지 마세요.** 그러면 검색이 너무 쉽게 맞혀 점수가 실제보다 높게 나옵니다(5번에서 이것도 잽니다).

**세 가지 조건**을 지켜야 합니다.

1. **정답이 2개 이상인 문항이 2개 이상** — 답이 한 문서에만 있는 질문만 모으면 Recall 이 Hit 과 늘 같은 값이 되어, 지표를 넷 만들어 놓고 사실은 둘만 보게 됩니다.
2. **정답이 4개 이상인 '넓은 질문'이 정확히 1개** — 여러 문서에 답이 흩어진 질문입니다. 6번에서 이 문항이 무엇을 망가뜨리는지 보고 **직접 고칠** 것입니다.
3. **같은 질문을 두 번 넣지 않기**

**예시** (형식만 보여 주는 것입니다 — 질문은 직접 쓰세요)

```
query_id  query                          gold_ids   evidence
q1        (질문)                          h04|h05    (h04 의 문장) || (h05 의 문장)
q2        (질문)                          h15        (h15 의 문장)
```

<details><summary>힌트</summary>

```text
접근방법:
- 먼저 표를 눈으로 훑으며 '이건 물어볼 만하다' 싶은 것을 고른다.
- 답이 두 문서에 걸치는 질문은 억지로 만들지 말고, 서로 이어진 항목에서 자연스럽게 찾는다.
- 근거 문장은 눈으로 옮겨 적지 말고 본문에서 복사한다.

세부구현:
1. 전체 본문을 한 번 출력해 읽는다. 표가 잘려 보이면 한 행씩 찍어 본다.
2. 질문마다 (질문, 정답 id 목록, 근거 문장 목록) 세 쪽짜리 자료를 목록으로 모은다.
3. 그 목록을 돌면서 id 목록은 파이프로, 근거 목록은 공백 파이프 파이프 공백으로 이어 붙인다.
   3-1. 이어 붙이는 순서가 서로 어긋나지 않게 같은 반복 안에서 처리한다.
4. 만든 행들을 DataFrame 으로 만들고 열 순서를 요구사항대로 맞춘다.
```

</details>

> 아래 셀을 먼저 실행해 **문서를 다 읽고** 시작하세요. 읽지 않고 쓴 질문은 5번 점검에서 걸립니다.

In [ ]:
# 24건을 전부 읽습니다 -- 질문은 이 안에서 나옵니다.
#  1번에서 만든 faq 를 그대로 쓰되, 아직 없으면 여기서 읽습니다(이 셀만 따로 실행해도 되게).
if 'faq' not in dir():
    import pandas as pd

    faq = pd.read_csv('data/hr_faq.csv')

for row in faq.itertuples():
    print(f'[{row.id}] ({row.분류}) {row.제목}')
    print('   ', row.본문)
    print()

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert list(my_eval.columns) == ['query_id', 'query', 'gold_ids', 'evidence'], '열 이름과 순서를 맞춰 주세요'
assert len(my_eval) >= 8, '질문을 8개 이상 만드세요'
assert my_eval['query'].duplicated().sum() == 0, '같은 질문이 두 번 들어 있습니다'

gold_lists = [g.split('|') for g in my_eval['gold_ids']]
evidence_lists = [e.split(' || ') for e in my_eval['evidence']]
assert all(len(g) == len(e) for g, e in zip(gold_lists, evidence_lists)), \
    '정답 id 개수와 근거 문장 개수가 다른 문항이 있습니다'
assert sum(1 for g in gold_lists if len(g) >= 2) >= 2, '정답이 2개 이상인 문항이 2개 이상 필요합니다'
assert sum(1 for g in gold_lists if len(g) >= 4) == 1, "정답이 4개 이상인 '넓은 질문' 이 정확히 1개 필요합니다"
print('✅ 통과!')

## 5. 내보내기 전 점검 — 이 평가셋을 믿어도 되나

**배경**: 평가셋에 결함이 있으면 **지표가 조용히 거짓말을 합니다.** 숫자는 멀쩡하게 나오는데 그 숫자가 아무것도 말해 주지 않습니다. 그래서 쓰기 전에 기계로 한 번 훑습니다.

| 점검 | 통과 못 하면 |
|---|---|
| 1. 정답 문서 id 가 코퍼스에 실제로 있는가 | 그 문항은 영원히 0점이 되는데 **에러는 안 난다** |
| 2. 근거 문장이 그 문서 원문에 있는가 | 근거를 믿을 수 없다 - 지어냈거나 옮겨 적다 달라진 것이다 |
| 3. 질문이 원문을 베끼지 않았는가 | 점수가 실제보다 높게 나온다 |
| 4. 한 문서가 여러 문항의 정답으로 쓰이는가 | 결함은 아니지만, 그 문서만 잘 찾아도 점수가 오른다 |

네 가지 모두 **기계가 볼 수 있는 것**입니다. 기계는 근거가 원문에 있는지는 알아도 그 문장이 **질문의 답인지는 모릅니다.** 그래서 점검이 전부 통과해도 그것은 '명백한 결함이 없다' 는 뜻일 뿐입니다 — 마지막 확인은 사람이 합니다(자가채점 뒤에 그 셀이 있습니다).

**요구사항**: 아래 제공 셀을 실행한 뒤 네 가지를 확인하세요.

- **`bad_ids`** — 코퍼스에 없는 정답 id 의 리스트. 비어 있어야 정상입니다.
- **`bad_evidence`** — 근거가 원문에 없는 것의 리스트. 원소는 **`(query_id, doc_id)` 튜플**입니다. 비어 있어야 정상입니다.
- **`check`** — 문항별 점검 표 DataFrame. 열 이름과 순서는 **`query_id`, `겹침`** 이고, `겹침` 은 **그 문항의 정답 문서들과의 겹침 평균**입니다. 만든 뒤 `0.6` 을 넘는 문항이 몇 개인지 출력하세요. (원본 `my_eval` 에 열을 붙이지 말고 **따로** 만드세요 — 뒤 문제들이 `my_eval` 을 그대로 씁니다.)
- **`shared`** — 두 번 이상 정답으로 쓰인 문서 id 를 모은 리스트(정렬). 결함이 아니므로 출력만 하면 됩니다.

**예시**

```
없는 문서 id  : []
원문에 없는 근거: []
겹침이 0.6 을 넘는 문항: 0개
두 번 이상 쓰인 정답 문서: ['h01', 'h04']
```

<details><summary>힌트</summary>

```text
접근방법:
- 문항을 한 번 돌면서 정답 id 와 근거 문장을 짝지어 검사하면 1번과 2번이 함께 끝난다.
- 겹침은 한 문항의 정답 문서마다 재서 평균을 낸다.

세부구현:
1. 코퍼스의 id 를 집합으로 한 번 모아 둔다. 문항마다 표를 뒤지면 느리다.
2. 문항을 돌며 정답 id 와 근거 문장을 짝지어 반복한다.
   2-1. id 가 집합에 없으면 그 id 를 첫 목록에 넣는다.
   2-2. id 는 있는데 근거가 원문에 없으면 문항 번호와 id 를 짝으로 두 번째 목록에 넣는다.
3. 겹침은 문항의 정답 문서마다 재서 평균을 내고 새 열로 붙인다.
4. 모든 정답 id 를 한 목록에 펼친 뒤 두 번 이상 나오는 것만 중복 없이 모은다.
```

</details>

In [ ]:
# [제공 코드] 질문이 원문을 얼마나 베꼈는지 재는 함수 -- 이 셀은 실행만 하세요.
#  질문을 세 글자씩 잘라 그 덩어리가 문서 본문에 그대로 나오는 비율을 봅니다.
#  두 글자로 세면 한국어 조사·어미가 자주 겹쳐 베끼지 않은 질문까지 걸립니다.
import re


def overlap_ratio(query, text):
    # 띄어쓰기만 다른 표현도 같은 것으로 세도록 공백을 모두 지운다
    squeezed = re.sub(r'\s+', '', query)
    # 세 글자씩 한 칸씩 밀며 잘라 중복 없이 모은다
    grams = {squeezed[i:i + 3] for i in range(len(squeezed) - 2)}
    body_text = re.sub(r'\s+', '', text)
    return sum(1 for g in grams if g in body_text) / len(grams)


print('겹침 측정 함수 준비 완료')

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert bad_ids == [], f'코퍼스에 없는 문서 id 입니다: {bad_ids}'
assert bad_evidence == [], f'근거 문장이 그 문서 원문에 없습니다: {bad_evidence}'
assert list(check.columns) == ['query_id', '겹침'], 'check 의 열 이름과 순서를 맞춰 주세요'
assert len(check) == len(my_eval), '모든 문항이 점검 표에 들어가야 합니다'
assert (check['겹침'] > 0.6).sum() == 0, '원문을 그대로 베낀 질문이 있습니다 - 물어볼 법한 말로 고쳐 주세요'
assert isinstance(shared, list), 'shared 는 문서 id 리스트입니다'
print('✅ 통과!')

### ✋ 마지막 점검은 사람이

위 네 가지는 **기계가 볼 수 있는 것**만 봤습니다. 남은 하나 — **그 근거 문장이 정말 그 질문의 답인가** — 는 읽어야 압니다. 아래 셀을 실행해 문항마다 질문과 근거를 나란히 놓고 확인하세요. 답이 아닌 문장이 보이면 4번으로 돌아가 그 문항의 `gold_ids` 와 `evidence` 를 고칩니다.

In [ ]:
# [제공 코드] 사람이 하는 마지막 점검 -- 실행하고 눈으로 읽으세요.
#  quoted_in_faq 는 '이 문장이 원문에 있나' 까지만 봅니다. '이 질문의 답인가' 는 못 봅니다.
#  4번에서 만든 my_eval 을 그대로 씁니다(아직 만들지 않았으면 안내만 하고 넘어갑니다).
if 'my_eval' not in dir():
    print('4번에서 my_eval 을 먼저 만든 뒤 이 셀을 실행하세요.')
else:
    for row in my_eval.itertuples():
        print(f'[{row.query_id}] {row.query}')
        # 정답 id 와 근거 문장은 같은 순서로 이어 붙였으므로 zip 으로 짝지어 읽는다
        for doc_id, sentence in zip(row.gold_ids.split('|'), row.evidence.split(' || ')):
            print(f'   {doc_id} :: {sentence}')
        print()

## 6. 내 평가셋으로 검색기 재기 — 그리고 평가셋을 고치기

**배경**: 평가셋이 준비됐으니 이제 **재 봅니다.** 그런데 이 과제의 진짜 목적은 점수를 내는 것이 아닙니다. 점수를 읽고 **어디가 문제인지 찾아 고치는 것**입니다. 낮은 점수가 검색기 탓인지 평가셋 탓인지 가릴 줄 알아야 그다음에 무엇을 고칠지 정할 수 있습니다.

세 단계로 갑니다. **1단계** K 별로 재서 표를 만들고, **2단계** 문항별로 쪼개 보아 어느 문항이 표를 끌어내리는지 찾고, **3단계** 그 문항을 고쳐 다시 잽니다.

> 아래 제공 셀의 지표 네 개는 **RAG 평가 과제에서 여러분이 직접 만든 것과 같은 함수**입니다. 여기서는 만드는 것이 아니라 **쓰는** 것이 목적이라 그대로 드립니다.

In [ ]:
# [제공 코드] 검색 평가 지표 네 개 — 이 셀은 실행만 하세요.
#  RAG 평가 과제에서 여러분이 직접 만든 것과 같은 함수입니다. 여기서는 도구로 씁니다.
#  네 함수의 인자는 모두 같습니다: ranked(검색 결과 id 를 1위부터), gold(정답 id 목록), k.
def hit_at_k(ranked, gold, k):
    # 상위 k개 안에 정답이 하나라도 있으면 1.0 -- 맞혔나 못 맞혔나만 본다
    return 1.0 if any(r in gold for r in ranked[:k]) else 0.0


def precision_at_k(ranked, gold, k):
    # 꺼내 온 k개 중 몇 개가 정답이었나 -- 나누는 수는 결과 길이가 아니라 언제나 k
    return sum(1 for r in ranked[:k] if r in gold) / k


def recall_at_k(ranked, gold, k):
    # 정답 전체 중 몇 개를 건졌나 -- 나누는 수가 정답 개수라 정답이 많으면 낮아진다
    return sum(1 for r in ranked[:k] if r in gold) / len(gold)


def mrr_at_k(ranked, gold, k):
    # 첫 정답이 몇 위였나 -- 1위면 1, 2위면 0.5. 순위를 보는 유일한 지표다
    for rank, doc_id in enumerate(ranked[:k], 1):
        if doc_id in gold:
            return 1 / rank
    return 0.0


print('지표 네 개 준비 완료')

### 1단계 — K 를 바꿔 가며 표 만들기

**요구사항**: `K = 1, 3, 5, 10` 각각에 대해 여러분의 평가셋 전체를 네 지표로 재어 DataFrame **`k_table`** 을 만드세요.

- 열 이름과 순서는 **`K`, `Hit`, `Precision`, `Recall`, `MRR`** 이고, 행은 K 가 작은 것부터입니다.
- 각 값은 **전체 문항의 평균**이고 **소수 셋째 자리까지 반올림**합니다.
- 문항마다 검색은 **한 번만** 하세요. 지표마다 다시 검색하면 느리고, 재는 대상도 흔들립니다.
- 만든 표를 `display()` 로 보세요.

**예시** (모범답안 8문항 기준 — 여러분의 값은 다릅니다)

```
   K    Hit  Precision  Recall    MRR
   1  0.875      0.875   0.688  0.875
   3  1.000      0.458   0.863  0.938
```

<details><summary>힌트</summary>

```text
접근방법:
- 바깥 반복은 K, 안쪽 반복은 문항이다.
- 문항마다 검색 결과를 한 번 받아 네 지표에 모두 넘긴다.

세부구현:
1. K 값들을 담은 목록을 만들고 결과를 모을 빈 목록을 준비한다.
2. K 마다 문항을 돌며 검색 결과와 정답 목록을 얻는다.
   2-1. 정답 목록은 파이프로 나눈 것이다.
   2-2. 네 지표를 각각 재어 한 문항의 값 네 개를 모은다.
3. 문항들의 값을 지표별로 평균 내고 반올림해 한 행으로 만든다.
4. 행들을 DataFrame 으로 만든다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert list(k_table.columns) == ['K', 'Hit', 'Precision', 'Recall', 'MRR'], '열 이름과 순서를 맞춰 주세요'
assert list(k_table['K']) == [1, 3, 5, 10], 'K 는 1, 3, 5, 10 순서입니다'
for name in ['Hit', 'Precision', 'Recall', 'MRR']:
    assert k_table[name].between(0, 1).all(), f'{name} 은 0 과 1 사이여야 합니다'

# K 를 키우면 더 넓게 보는 것이므로 Hit·Recall·MRR 은 절대 줄어들 수 없다.
#  줄었다면 상위 k개를 자르지 않았거나 정답 목록을 잘못 만든 것이다.
for name in ['Hit', 'Recall', 'MRR']:
    values = list(k_table[name])
    assert all(a <= b for a, b in zip(values, values[1:])), \
        f'{name} 이 K 가 커지는데 줄었습니다 - 상위 k개만 보고 있는지 확인하세요'
print('✅ 통과!')

### 2단계 — 어느 문항이 표를 끌어내리는가

**배경**: 평균은 원인을 감춥니다. `Recall` 이 낮게 나왔다면 검색기가 못 찾은 것일 수도 있고, **질문이 너무 넓어 애초에 다 찾을 수 없는 것**일 수도 있습니다. 문항별로 쪼개 보면 갈립니다.

**요구사항**: `K=3` 으로 문항마다 재어 DataFrame **`per_item`** 을 만드세요.

- 열 이름과 순서는 **`query_id`, `정답수`, `Hit3`, `Recall3`** 입니다.
- `정답수` 는 그 문항의 정답 문서 개수, `Hit3`·`Recall3` 은 K=3 으로 잰 값(소수 셋째 자리 반올림)입니다.
- `Recall3` 이 낮은 것부터 보이도록 **`Recall3` 오름차순으로 정렬**해 `display()` 하세요.

그리고 아래 서술형 답안 셀에 답하세요.

**질문**: 정답이 4개 이상인 그 '넓은 질문'의 `Recall3` 을 보세요. 이 값이 **낮을 수밖에 없는 이유**는 무엇인가요? 검색기를 아무리 잘 고쳐도 이 문항의 `Recall3` 이 넘을 수 없는 한계값이 있습니다. 그 값은 얼마이고 왜 그런가요?

<details><summary>힌트</summary>

```text
접근방법:
- 1단계와 같은 계산인데 평균을 내지 않고 문항별로 남긴다.

세부구현:
1. 문항을 돌며 정답 목록과 검색 결과를 얻는다.
2. 문항 번호, 정답 개수, 두 지표 값을 한 행으로 모은다.
3. DataFrame 으로 만든 뒤 Recall 열 기준으로 오름차순 정렬한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert list(per_item.columns) == ['query_id', '정답수', 'Hit3', 'Recall3'], '열 이름과 순서를 맞춰 주세요'
assert len(per_item) == len(my_eval), '모든 문항이 들어가야 합니다'

# 상위 3개만 보는데 정답이 5개라면 Recall 은 아무리 잘해도 3/5 를 넘을 수 없다.
#  이 관계는 어떤 평가셋에서도 성립하므로, 깨졌다면 계산이 틀린 것이다.
for row in per_item.itertuples():
    assert row.Recall3 <= min(1.0, 3 / row.정답수) + 1e-9, \
        f'{row.query_id}: 상위 3개로는 나올 수 없는 Recall 입니다 - 정답 목록을 확인하세요'
print('✅ 통과!')

**서술형 답안**

*(여기에 자신의 답을 서술하세요)*

### 3단계 — 질문을 고쳐 다시 재기

**배경**: 2단계에서 찾은 그 넓은 질문은 사실 **여러 질문을 하나로 묶은 것**입니다. 쪼개면 각 문항의 정답이 줄고, 그러면 `Recall` 이 다시 **검색기의 성능을 재는 숫자**가 됩니다.

**요구사항**:
- 그 넓은 질문을 **주제가 다른 두 질문으로 쪼개** 새 평가셋 **`my_eval2`** 를 만드세요. 원래 문항은 **빼고** 쪼갠 둘을 **넣습니다**(나머지 문항은 그대로).
  - 열 이름과 순서는 `my_eval` 과 같은 **`query_id`, `query`, `gold_ids`, `evidence`** 입니다.
  - 쪼갠 두 문항의 `query_id` 는 기존과 겹치지 않게 붙이세요(예: `q1a`, `q1b`).
  - 근거 문장은 여기서도 **원문 그대로**입니다.
- `my_eval2` 로 **`k_table2`** 를 만드세요. 만드는 방법은 1단계와 같습니다(열도 같습니다).
- 두 표의 **K=3 행 `Recall`** 을 나란히 출력해 얼마나 달라졌는지 보세요.

그리고 아래 서술형 답안 셀에 답하세요.

**질문**: `Recall` 이 올라갔습니다. 그렇다면 **검색기가 좋아진 것인가요?** 아니라면 이 작업으로 실제로 좋아진 것은 무엇인가요?

<details><summary>힌트</summary>

```text
접근방법:
- 평가셋을 다시 만드는 것이지 검색을 바꾸는 것이 아니다.
- 표 만드는 코드는 1단계 것을 그대로 쓰되 대상만 바꾼다.

세부구현:
1. 기존 문항에서 넓은 질문만 뺀 목록을 만든다.
2. 쪼갠 두 문항을 같은 열 구조로 만들어 이어 붙인다.
3. 1단계와 같은 방법으로 K 별 표를 만든다.
4. 두 표에서 K 가 3 인 행의 Recall 값을 꺼내 함께 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert list(my_eval2.columns) == ['query_id', 'query', 'gold_ids', 'evidence'], '열 이름과 순서를 맞춰 주세요'
assert len(my_eval2) == len(my_eval) + 1, '넓은 질문 1개를 빼고 2개를 넣었으므로 문항이 하나 늘어야 합니다'
assert my_eval2['query_id'].is_unique, 'query_id 가 겹칩니다'

gold2 = [g.split('|') for g in my_eval2['gold_ids']]
assert max(len(g) for g in gold2) < 4, '정답이 4개 이상인 문항이 아직 남아 있습니다'
assert all(len(g) == len(e.split(' || ')) for g, e in zip(gold2, my_eval2['evidence'])), \
    '쪼갠 문항의 정답 id 개수와 근거 문장 개수가 다릅니다'
# 새로 적은 근거도 원문에 있어야 한다 -- 쪼개면서 요약해 적기 쉬운 자리다
for row in my_eval2.itertuples():
    for doc_id, sentence in zip(row.gold_ids.split('|'), row.evidence.split(' || ')):
        assert quoted_in_faq(sentence, doc_id), f'{row.query_id} 의 근거가 {doc_id} 원문에 없습니다'

assert list(k_table2.columns) == list(k_table.columns), 'k_table2 는 k_table 과 같은 열 구조입니다'
for name in ['Hit', 'Recall', 'MRR']:
    values = list(k_table2[name])
    assert all(a <= b for a, b in zip(values, values[1:])), f'{name} 이 K 가 커지는데 줄었습니다'
print('✅ 통과!')

**서술형 답안**

*(여기에 자신의 답을 서술하세요)*

---
수고했어요! 이 과제에서 여러분은 **재는 도구를 직접 만들었습니다.**

| 한 일 | 남길 것 |
|---|---|
| 한 행이 한 문서인 코퍼스를 색인 | 자를 것이 없으면 라벨이 문서 id 그대로다 |
| 질문을 쓰고 정답 라벨과 근거 문장을 붙임 | 근거 문장이 있어야 나중에 그 라벨을 검수할 수 있다 |
| 내보내기 전에 기계로 점검 | 없는 id·지어낸 근거·베낀 질문은 **에러 없이** 지표를 망친다 |
| 점검을 통과한 라벨을 눈으로 검수 | 인용 대조는 **근거가 원문에 있다**까지만 보증한다 - 답인지는 사람이 읽는다 |
| 문항별로 쪼개 원인을 찾음 | `Recall` 은 **정답 개수와 함께** 읽는다 |
| 질문을 고쳐 다시 잼 | 점수가 오른 것과 **검색기가 좋아진 것**은 다른 이야기다 |

한 가지만 남긴다면: **평가셋이 틀리면 그것으로 잰 점수가 전부 함께 틀립니다.** 그런데 화면에는 멀쩡해 보이는 숫자가 찍히기 때문에 알아채기가 어렵습니다. 그래서 재기 전에 재는 도구부터 검수합니다.

> 라벨을 **모델에게 시키고** 그 판정을 검증하는 방법이 궁금하다면 `부록_평가셋_구축.ipynb` 를 보세요. 이 과제에서 손으로 한 일을 모델에게 **초안으로** 시키고, 인용을 기계로 대조한 뒤 **확정은 사람이 하는** 절차입니다.